# 05 — Player Value Analysis

This notebook turns the modelling outputs from **xT** and **VAEP-style action valuation** into a player-centric analytics layer.

## Objectives

1. Load the outputs from:
   - Notebook 02 (`actions.parquet`)
   - Notebook 03 (`actions_xt.parquet`, `player_xt_summary.parquet`)
   - Notebook 04 (`actions_vaep.parquet`, `player_vaep_summary.parquet`)
2. Build an integrated **player value table** combining:
   - volume
   - xT contribution
   - VAEP contribution
   - progression profile
3. Normalise player impact metrics using:
   - contribution per 100 actions
   - contribution per 90 equivalent proxy
   - percentile ranks
4. Create player segments for scouting and interpretation.
5. Produce visual comparisons between:
   - xT vs VAEP
   - volume vs efficiency
   - progression vs overall value
6. Persist the final player analytics table for downstream reporting.

## Outputs

- `data/processed/player_value_table.parquet`
- `data/processed/player_segments.parquet`
- `db/football_value.duckdb`
  - `player_value_table`
  - `player_segments`


## 0. Setup

In [ ]:
from __future__ import annotations

import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
warnings.filterwarnings("ignore")


In [ ]:
try:
    import duckdb
except Exception as e:
    raise ImportError("duckdb is required. Install it with: pip install duckdb") from e

try:
    from sklearn.preprocessing import StandardScaler
    from sklearn.cluster import KMeans
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False


## 1. Project paths

In [ ]:
def find_repo_root(start: Optional[Path] = None) -> Path:
    start = start or Path.cwd()
    for p in [start, *start.parents]:
        if (p / "README.md").exists():
            return p
    return start

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
DB_DIR = REPO_ROOT / "db"
REPORTS_DIR = REPO_ROOT / "reports"

for d in [PROCESSED_DIR, DB_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DB_PATH = DB_DIR / "football_value.duckdb"

ACTIONS_PATH = PROCESSED_DIR / "actions.parquet"
ACTIONS_XT_PATH = PROCESSED_DIR / "actions_xt.parquet"
PLAYER_XT_PATH = PROCESSED_DIR / "player_xt_summary.parquet"
ACTIONS_VAEP_PATH = PROCESSED_DIR / "actions_vaep.parquet"
PLAYER_VAEP_PATH = PROCESSED_DIR / "player_vaep_summary.parquet"

PLAYER_VALUE_PATH = PROCESSED_DIR / "player_value_table.parquet"
PLAYER_SEGMENTS_PATH = PROCESSED_DIR / "player_segments.parquet"

REPO_ROOT, DB_PATH


## 2. Configuration

In [ ]:
@dataclass(frozen=True)
class Config:
    min_actions_per_player: int = 150
    possession_minutes_per_match: float = 96.0
    save_outputs: bool = True
    n_clusters: int = 4
    random_state: int = 42

CFG = Config()
CFG


## 3. Load modelling outputs

Expected inputs:
- `actions.parquet`
- `actions_xt.parquet`
- `player_xt_summary.parquet`
- `actions_vaep.parquet`
- `player_vaep_summary.parquet`


In [ ]:
required_paths = [
    ACTIONS_PATH,
    ACTIONS_XT_PATH,
    PLAYER_XT_PATH,
    ACTIONS_VAEP_PATH,
    PLAYER_VAEP_PATH,
]

missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Missing required inputs. Run Notebooks 02, 03 and 04 first:\n" + "\n".join(missing_paths)
    )

actions = pd.read_parquet(ACTIONS_PATH)
actions_xt = pd.read_parquet(ACTIONS_XT_PATH)
player_xt_summary = pd.read_parquet(PLAYER_XT_PATH)
actions_vaep = pd.read_parquet(ACTIONS_VAEP_PATH)
player_vaep_summary = pd.read_parquet(PLAYER_VAEP_PATH)

print("actions:", actions.shape)
print("actions_xt:", actions_xt.shape)
print("player_xt_summary:", player_xt_summary.shape)
print("actions_vaep:", actions_vaep.shape)
print("player_vaep_summary:", player_vaep_summary.shape)


## 4. Build base player volume table

In [ ]:
player_base = (
    actions.groupby("player", dropna=False)
    .agg(
        n_actions=("event_id", "count"),
        n_matches=("match_id", "nunique"),
        n_teams=("team", "nunique"),
        primary_team=("team", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
        passes=("is_pass", "sum"),
        carries=("is_carry", "sum"),
        dribbles=("is_dribble", "sum"),
        shots=("is_shot", "sum"),
        progressive_actions=("is_progressive_action", "sum"),
    )
    .reset_index()
)

player_base = player_base[player_base["player"].notna()].copy()

player_base["actions_per_match"] = np.where(
    player_base["n_matches"] > 0,
    player_base["n_actions"] / player_base["n_matches"],
    np.nan
)

player_base["pseudo_minutes"] = player_base["n_matches"] * CFG.possession_minutes_per_match

player_base["actions_per_90"] = np.where(
    player_base["pseudo_minutes"] > 0,
    90 * player_base["n_actions"] / player_base["pseudo_minutes"],
    np.nan
)

player_base["progressive_actions_per_90"] = np.where(
    player_base["pseudo_minutes"] > 0,
    90 * player_base["progressive_actions"] / player_base["pseudo_minutes"],
    np.nan
)

player_base = player_base[player_base["n_actions"] >= CFG.min_actions_per_player].copy()
player_base.head()


## 5. Merge xT and VAEP summaries

In [ ]:
player_value = (
    player_base
    .merge(
        player_xt_summary.rename(columns={
            "total_xt_added": "xt_total",
            "avg_xt_added": "xt_mean",
            "xt_added_per_100_actions": "xt_per_100_actions",
            "n_xt_actions": "xt_actions",
        }),
        on="player",
        how="left"
    )
    .merge(
        player_vaep_summary.rename(columns={
            "total_vaep": "vaep_total",
            "mean_vaep": "vaep_mean",
            "vaep_per_100_actions": "vaep_per_100_actions",
            "total_p_score": "p_score_total",
            "total_p_concede": "p_concede_total",
        }),
        on="player",
        how="left"
    )
)

player_value.head()


## 6. Add derived contribution metrics

In [ ]:
player_value["xt_per_90"] = np.where(
    player_value["pseudo_minutes"] > 0,
    90 * player_value["xt_total"].fillna(0) / player_value["pseudo_minutes"],
    np.nan
)

player_value["vaep_per_90"] = np.where(
    player_value["pseudo_minutes"] > 0,
    90 * player_value["vaep_total"].fillna(0) / player_value["pseudo_minutes"],
    np.nan
)

player_value["pass_share"] = np.where(player_value["n_actions"] > 0, player_value["passes"] / player_value["n_actions"], np.nan)
player_value["carry_share"] = np.where(player_value["n_actions"] > 0, player_value["carries"] / player_value["n_actions"], np.nan)
player_value["dribble_share"] = np.where(player_value["n_actions"] > 0, player_value["dribbles"] / player_value["n_actions"], np.nan)
player_value["shot_share"] = np.where(player_value["n_actions"] > 0, player_value["shots"] / player_value["n_actions"], np.nan)

player_value["vaep_minus_xt"] = player_value["vaep_per_100_actions"].fillna(0) - player_value["xt_per_100_actions"].fillna(0)

player_value["xt_per_progressive_action"] = np.where(
    player_value["progressive_actions"] > 0,
    player_value["xt_total"].fillna(0) / player_value["progressive_actions"],
    np.nan
)

player_value["vaep_per_progressive_action"] = np.where(
    player_value["progressive_actions"] > 0,
    player_value["vaep_total"].fillna(0) / player_value["progressive_actions"],
    np.nan
)

player_value.head()


## 7. Percentile ranks and composite scouting indicators

In [ ]:
percentile_cols = [
    "actions_per_90",
    "progressive_actions_per_90",
    "xt_per_90",
    "vaep_per_90",
    "xt_per_100_actions",
    "vaep_per_100_actions",
]

for col in percentile_cols:
    player_value[f"{col}_pct"] = player_value[col].rank(pct=True)

player_value["progression_score"] = (
    0.50 * player_value["progressive_actions_per_90_pct"]
    + 0.50 * player_value["xt_per_90_pct"]
)

player_value["overall_value_score"] = (
    0.35 * player_value["xt_per_90_pct"]
    + 0.45 * player_value["vaep_per_90_pct"]
    + 0.20 * player_value["actions_per_90_pct"]
)

player_value["efficiency_score"] = (
    0.40 * player_value["xt_per_100_actions_pct"]
    + 0.60 * player_value["vaep_per_100_actions_pct"]
)

player_value["scouting_score"] = (
    0.35 * player_value["overall_value_score"]
    + 0.35 * player_value["progression_score"]
    + 0.30 * player_value["efficiency_score"]
)

player_value = player_value.sort_values("scouting_score", ascending=False).reset_index(drop=True)
player_value.head(15)


## 8. Player segmentation

In [ ]:
def assign_rule_based_segment(row):
    if row["overall_value_score"] >= 0.90 and row["actions_per_90_pct"] >= 0.75:
        return "Elite high-volume creator"
    if row["efficiency_score"] >= 0.85 and row["actions_per_90_pct"] < 0.75:
        return "Efficient value creator"
    if row["progression_score"] >= 0.85 and row["xt_per_90_pct"] >= 0.80:
        return "Progression specialist"
    if row["overall_value_score"] >= 0.65:
        return "Balanced contributor"
    return "Support profile"

player_value["rule_based_segment"] = player_value.apply(assign_rule_based_segment, axis=1)
player_value["rule_based_segment"].value_counts()


In [ ]:
if SKLEARN_AVAILABLE:
    cluster_features = [
        "actions_per_90",
        "progressive_actions_per_90",
        "xt_per_90",
        "vaep_per_90",
        "pass_share",
        "carry_share",
        "dribble_share",
        "shot_share",
    ]

    cluster_df = player_value[cluster_features].fillna(0).copy()
    scaler = StandardScaler()
    X_cluster = scaler.fit_transform(cluster_df)

    kmeans = KMeans(n_clusters=CFG.n_clusters, random_state=CFG.random_state, n_init=10)
    player_value["cluster_id"] = kmeans.fit_predict(X_cluster)
else:
    player_value["cluster_id"] = -1

player_value[["player", "rule_based_segment", "cluster_id"]].head(15)


## 9. Build compact player segments table

In [ ]:
player_segments = player_value[[
    "player",
    "primary_team",
    "n_actions",
    "n_matches",
    "actions_per_90",
    "progressive_actions_per_90",
    "xt_per_90",
    "vaep_per_90",
    "xt_per_100_actions",
    "vaep_per_100_actions",
    "scouting_score",
    "rule_based_segment",
    "cluster_id",
]].copy()

player_segments = player_segments.sort_values("scouting_score", ascending=False).reset_index(drop=True)
player_segments.head(20)


## 10. xT vs VAEP comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(
    player_value["xt_per_90"],
    player_value["vaep_per_90"],
    alpha=0.7
)

ax.set_title("Player Value Map: xT per 90 vs VAEP per 90")
ax.set_xlabel("xT per 90")
ax.set_ylabel("VAEP per 90")

top_players = player_value.head(12)
for _, r in top_players.iterrows():
    ax.annotate(r["player"], (r["xt_per_90"], r["vaep_per_90"]), fontsize=8)

plt.show()


## 11. Volume vs efficiency map

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(
    player_value["actions_per_90"],
    player_value["vaep_per_100_actions"],
    alpha=0.7
)

ax.set_title("Player Value Map: Volume vs VAEP Efficiency")
ax.set_xlabel("Actions per 90")
ax.set_ylabel("VAEP per 100 actions")

top_players = player_value.head(12)
for _, r in top_players.iterrows():
    ax.annotate(r["player"], (r["actions_per_90"], r["vaep_per_100_actions"]), fontsize=8)

plt.show()


## 12. Top player rankings

In [ ]:
top_total_value = player_value.sort_values("vaep_total", ascending=False)[[
    "player", "primary_team", "n_actions", "xt_total", "vaep_total", "xt_per_90", "vaep_per_90", "scouting_score", "rule_based_segment"
]].head(20)

top_total_value


In [ ]:
top_efficiency = player_value.sort_values("vaep_per_100_actions", ascending=False)[[
    "player", "primary_team", "n_actions", "xt_per_100_actions", "vaep_per_100_actions", "scouting_score", "rule_based_segment"
]].head(20)

top_efficiency


In [ ]:
top_progression = player_value.sort_values("progression_score", ascending=False)[[
    "player", "primary_team", "progressive_actions_per_90", "xt_per_90", "progression_score", "rule_based_segment"
]].head(20)

top_progression


## 13. Team-level context from player table

In [ ]:
team_context = (
    player_value.groupby("primary_team", dropna=False)
    .agg(
        n_players=("player", "count"),
        team_xt_total=("xt_total", "sum"),
        team_vaep_total=("vaep_total", "sum"),
        team_progressive_actions=("progressive_actions", "sum"),
        avg_scouting_score=("scouting_score", "mean"),
    )
    .reset_index()
    .sort_values("team_vaep_total", ascending=False)
)

team_context.head(20)


## 14. Optional radar-style comparison

In [ ]:
def radar_plot(player_names, metrics, df):
    subset = df[df["player"].isin(player_names)].copy()
    if subset.empty or len(metrics) < 3:
        print("Not enough data for radar plot.")
        return

    angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
    angles += angles[:1]

    fig = plt.figure(figsize=(8, 8))
    ax = plt.subplot(111, polar=True)

    for _, row in subset.iterrows():
        values = [row[m] for m in metrics]
        values += values[:1]
        ax.plot(angles, values, linewidth=2, label=row["player"])
        ax.fill(angles, values, alpha=0.10)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics)
    ax.set_title("Player Comparison Radar (percentile metrics)")
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.10))
    plt.show()

radar_metrics = [
    "actions_per_90_pct",
    "progressive_actions_per_90_pct",
    "xt_per_90_pct",
    "vaep_per_90_pct",
    "efficiency_score",
]

example_players = player_value["player"].head(3).tolist()
radar_plot(example_players, radar_metrics, player_value)


## 15. Save outputs

In [ ]:
if CFG.save_outputs:
    player_value.to_parquet(PLAYER_VALUE_PATH, index=False)
    player_segments.to_parquet(PLAYER_SEGMENTS_PATH, index=False)

    try:
        con.close()
    except:
        pass

    con = duckdb.connect(str(DB_PATH))
    con.execute("CREATE OR REPLACE TABLE player_value_table AS SELECT * FROM read_parquet(?)", [str(PLAYER_VALUE_PATH)])
    con.execute("CREATE OR REPLACE TABLE player_segments AS SELECT * FROM read_parquet(?)", [str(PLAYER_SEGMENTS_PATH)])
    con.close()

    print(f"Saved player value table to: {PLAYER_VALUE_PATH}")
    print(f"Saved player segments to:    {PLAYER_SEGMENTS_PATH}")
    print(f"Updated DuckDB at:           {DB_PATH}")


## 16. Preview from DuckDB

In [ ]:
con = duckdb.connect(str(DB_PATH))

summary_sql = '''
SELECT
    COUNT(*) AS n_players,
    AVG(xt_per_90) AS avg_xt_per_90,
    AVG(vaep_per_90) AS avg_vaep_per_90,
    AVG(scouting_score) AS avg_scouting_score,
    MAX(scouting_score) AS max_scouting_score
FROM player_value_table
'''
preview = con.execute(summary_sql).df()
con.close()

preview


# Conclusions

## Dataset Overview

The player value analysis is based on an event dataset containing:

* **158,114 on-ball actions**
* **441 players**
* **70 international matches**

After applying a minimum action threshold for analytical stability, the final analysis includes **334 players**, ensuring that the results are not dominated by extremely small sample sizes.

This provides a sufficiently large and diverse player universe for meaningful comparative analysis.

---

# Integrated Player Value Framework

This notebook integrates two complementary action valuation approaches:

**Expected Threat (xT)**
Measures the spatial progression value of ball movement.

**VAEP-style Action Value**
Estimates the change in scoring probability:

[
VAEP(action) = P(score) - P(concede)
]

Combining both frameworks allows the analysis to capture:

* spatial progression
* attacking contribution
* defensive risk
* contextual game state

This produces a richer player evaluation framework than either model alone.

---

# Distribution of Player Profiles

The rule-based segmentation produces the following player distribution:

| Segment                   | Players |
| ------------------------- | ------- |
| Support profile           | 231     |
| Balanced contributor      | 45      |
| Efficient value creator   | 33      |
| Elite high-volume creator | 14      |
| Progression specialist    | 11      |

This distribution reflects realistic football dynamics:

* most players provide **supporting contributions**
* a smaller group provides **consistent attacking value**
* only a few players generate **elite progression and attacking impact**

---

# High-Volume Elite Creators

The model identifies several players as **elite high-volume creators**, combining high action volume with strong attacking impact.

Examples include:

* **Neymar**
* **Mesut Özil**
* **Toni Kroos**
* **Kevin De Bruyne**
* **Philippe Coutinho**

These players combine:

* high action involvement
* strong progression metrics
* high VAEP contribution

Such profiles are typically associated with **creative midfielders and attacking playmakers**.

---

# Progression Specialists

The progression ranking highlights players with exceptional ball progression metrics.

Top progression profiles include:

* **Toni Kroos**
* **Mesut Özil**
* **Luka Modrić**
* **Joshua Kimmich**
* **Éver Banega**

These players exhibit high values for:

* **progressive actions per 90**
* **xT per 90**

indicating a strong role in advancing the ball into more dangerous areas.

---

# Efficiency-Based Value Creation

Efficiency rankings (VAEP per 100 actions) identify players who generate significant attacking impact with relatively fewer touches.

Examples include:

* **Dušan Tadić**
* **Timo Werner**
* **Yussuf Poulsen**
* **Luis Suárez Díaz**

These players convert possession into goal probability efficiently, even when their total action volume is lower.

---

# Volume vs Efficiency Trade-Off

The **Volume vs VAEP Efficiency map** reveals a clear trade-off between:

* **high involvement players**
* **high efficiency players**

Some players generate value through **constant involvement in possession**, while others create impact through **fewer but highly effective actions**.

This distinction is particularly important for recruitment and tactical analysis.

---

# Historical Validation

Several historically influential players appear near the top of the rankings:

* **Ronaldinho**
* **Lionel Messi**
* **Samuel Eto’o**
* **Andrés Iniesta**
* **Xavi Hernández**

This suggests that the action valuation framework captures meaningful football signal rather than random statistical patterns.

---

# Global Summary Metrics

Across the analysed player universe:

* **Average xT per 90:** 0.0355
* **Average VAEP per 90:** 8.22
* **Average scouting score:** 0.50

The maximum scouting score reaches **0.97**, indicating clear separation between elite contributors and the average player.

---

# Analytical Value for Recruitment

The player value table produced in this notebook enables several recruitment-oriented analyses:

* identification of high-impact creators
* detection of efficient value generators
* progression profiling
* role comparison across teams

The combination of **xT and VAEP metrics** provides a powerful framework for identifying players who consistently increase their team's scoring probability.

---

# Project Significance

At this stage, the project implements a complete football analytics pipeline:

```
StatsBomb event data
        ↓
Event modelling
        ↓
Action dataset
        ↓
Expected Threat (xT)
        ↓
VAEP-style action valuation
        ↓
Player value analysis
```

This structure mirrors the workflow used in modern football analytics environments.

---

# Next Step

The next stage of the project will analyse **team tactical styles**, focusing on how teams accumulate value through different patterns of:

* ball progression
* possession structure
* action profiles

This will extend the player-level analysis to the **team tactical level**.